# Phase 4 — Evaluate the improved retriever

Your pipeline is now:

Chunks → Embeddings → Hybrid Retrieval → Reranker → Top-5 results

We need to determine whether this actually improved over your baseline.

Do these next, in order:

- Calculate Top-1 / Top-3 / Top-5 for the new reranked results.
- Compare them against your existing baseline:
- Short: 116 queries
- Long: 41 queries
- Identify which queries improved, stayed the same, or became worse.
- Specifically check the previous failure categories:
- README/navigation noise — should now be gone.
- Vocabulary mismatch (PTO, vesting, etc.).
- Wrong neighboring policy.
- Correct document present but incorrectly ranked.
- Correct document missing from Top-5.
- If reranking improves the metrics, finalize it as your retrieval pipeline.
- If not, we test the next change rather than blindly adding more complexity.

## Imports

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

print("Libraries loaded.")

Libraries loaded.


## File paths

In [ ]:
BASE_DIR = Path(".")

BASELINE_SHORT =  BASE_DIR / "evaluation_results" / "retrieval_evaluation.json"
BASELINE_LONG = BASE_DIR / "evaluation_results" / "retrieval_evaluation_long.json"

IMPROVED_SHORT = BASE_DIR / "retrieval_improvement_outputs" / "short_queries_116_reranked_outputs.json"
IMPROVED_LONG = BASE_DIR / "retrieval_improvement_outputs" / "long_queries_41_reranked_outputs.json"

print("Files configured.")

Files configured.


## Load all four files

In [3]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


baseline_short = load_json(BASELINE_SHORT)
baseline_long = load_json(BASELINE_LONG)

improved_short = load_json(IMPROVED_SHORT)
improved_long = load_json(IMPROVED_LONG)
baseline_short_results = baseline_short["results"]
baseline_long_results = baseline_long["results"]

print("Baseline short queries:", len(baseline_short_results))
print("Baseline long queries:", len(baseline_long_results))
print("Improved short:", len(improved_short))
print("Improved long:", len(improved_long))

Baseline short queries: 116
Baseline long queries: 41
Improved short: 116
Improved long: 41


## Inspect improved structure

In [4]:
print(json.dumps(improved_short[0], indent=2, ensure_ascii=False)[:4000])

{
  "query_id": 1,
  "query": "How much vacation time do employees get?",
  "query_type": "short",
  "expected_document": "Vacation and Sick Leave.md",
  "results": [
    {
      "chunk_id": "chunk_000014",
      "document": "Vacation and Sick Leave.md",
      "file_path": "Benefits and Perks\\Vacation and Sick Leave.md",
      "category": "Benefits and Perks",
      "section_path": [
        "Vacation and Sick Leave"
      ],
      "section_title": "Vacation and Sick Leave",
      "chunk_index": 1,
      "word_count": 155,
      "character_count": 867,
      "content": "Taking time off and recharging is critical to doing your best work at Clef, so in addition to the recognized Holiday List, Clef offers 3 weeks (15 days) of paid vacation every year that accrues 1.25 of a day per month of work. Employees should schedule their vacations, let the rest of the team know, and add it to their shared work calendar at least a week in advance.\n\nEmployees also accrue 1 hour of sick leave for ev

## Evaluation function

This calculates Top-1 Accuracy, Top-3 Recall and Top-5 Recall based on whether the expected document appears in the retrieved results.

In [5]:
def evaluate_results(data):
    total = len(data)

    top1 = 0
    top3 = 0
    top5 = 0

    details = []

    for item in data:

        expected = item["expected_document"]

        results = item["results"]

        retrieved_documents = [
            result["document"]
            for result in results
        ]

        try:
            rank = retrieved_documents.index(expected) + 1
        except ValueError:
            rank = None

        if rank == 1:
            top1 += 1

        if rank is not None and rank <= 3:
            top3 += 1

        if rank is not None and rank <= 5:
            top5 += 1

        details.append({
            "query_id": item["query_id"],
            "query": item["query"],
            "expected_document": expected,
            "rank": rank
        })

    metrics = {
        "total_queries": total,
        "top1_accuracy": top1 / total * 100,
        "top3_recall": top3 / total * 100,
        "top5_recall": top5 / total * 100
    }

    return metrics, pd.DataFrame(details)

## Evaluate improved retrieval

In [6]:
improved_short_metrics, improved_short_details = evaluate_results(
    improved_short
)

improved_long_metrics, improved_long_details = evaluate_results(
    improved_long
)

print("SHORT QUERIES")
print(improved_short_metrics)

print("\nLONG QUERIES")
print(improved_long_metrics)

SHORT QUERIES
{'total_queries': 116, 'top1_accuracy': 82.75862068965517, 'top3_recall': 88.79310344827587, 'top5_recall': 91.37931034482759}

LONG QUERIES
{'total_queries': 41, 'top1_accuracy': 90.2439024390244, 'top3_recall': 97.5609756097561, 'top5_recall': 100.0}


## ## Evaluate your existing baseline

Your existing evaluation files may already contain metrics, but we'll calculate them again from the query-level results where possible.Corrected

In [7]:
def process_baseline_results(data):
    rows = []

    for i, item in enumerate(data, 1):

        expected = item["expected"]
        retrieved = item["retrieved"]

        try:
            rank = retrieved.index(expected) + 1
        except ValueError:
            rank = None

        rows.append({
            "query_id": i,
            "query": item["query"],
            "expected_document": expected,
            "rank": rank,
            "top1": item["top_1"],
            "top3": item["top_3"],
            "top5": item["top_5"]
        })

    return pd.DataFrame(rows)


baseline_short_details = process_baseline_results(
    baseline_short_results
)

baseline_long_details = process_baseline_results(
    baseline_long_results
)

display(baseline_short_details.head())

,query_id,query,expected_document,rank,top1,top3,top5
0,1,How much vacation time do employees get?,Vacation and Sick Leave.md,1.0,True,True,True
1,2,Can I take leave after having a baby?,New Parent Leave.md,1.0,True,True,True
2,3,What holidays does the company observe?,Holiday List.md,1.0,True,True,True
3,4,How many days of PTO do I earn per month?,Vacation and Sick Leave.md,1.0,True,True,True
4,5,"I'm not feeling well today, how does sick leav...",Vacation and Sick Leave.md,1.0,True,True,True


In [8]:
def process_improved_results(data):
    rows = []

    for item in data:

        expected = item["expected_document"]

        results = item["results"]

        retrieved_documents = [
            result["document"]
            for result in results
        ]

        try:
            rank = retrieved_documents.index(expected) + 1
        except ValueError:
            rank = None

        rows.append({
            "query_id": item["query_id"],
            "query": item["query"],
            "expected_document": expected,
            "rank": rank,
            "top1": rank == 1,
            "top3": rank is not None and rank <= 3,
            "top5": rank is not None and rank <= 5
        })

    return pd.DataFrame(rows)


improved_short_details = process_improved_results(
    improved_short
)

improved_long_details = process_improved_results(
    improved_long
)

display(improved_short_details.head())

,query_id,query,expected_document,rank,top1,top3,top5
0,1,How much vacation time do employees get?,Vacation and Sick Leave.md,1.0,True,True,True
1,2,Can I take leave after having a baby?,New Parent Leave.md,1.0,True,True,True
2,3,What holidays does the company observe?,Holiday List.md,1.0,True,True,True
3,4,How many days of PTO do I earn per month?,Vacation and Sick Leave.md,1.0,True,True,True
4,5,"I'm not feeling well today, how does sick leav...",Vacation and Sick Leave.md,1.0,True,True,True


In [9]:
def calculate_metrics(df):

    total = len(df)

    return {
        "total_queries": total,
        "top1_accuracy": df["top1"].mean() * 100,
        "top3_recall": df["top3"].mean() * 100,
        "top5_recall": df["top5"].mean() * 100
    }


baseline_short_metrics = calculate_metrics(
    baseline_short_details
)

baseline_long_metrics = calculate_metrics(
    baseline_long_details
)

improved_short_metrics = calculate_metrics(
    improved_short_details
)

improved_long_metrics = calculate_metrics(
    improved_long_details
)

print("Baseline Short:", baseline_short_metrics)
print("Improved Short:", improved_short_metrics)

print("\nBaseline Long:", baseline_long_metrics)
print("Improved Long:", improved_long_metrics)

Baseline Short: {'total_queries': 116, 'top1_accuracy': np.float64(78.44827586206897), 'top3_recall': np.float64(87.93103448275862), 'top5_recall': np.float64(89.65517241379311)}
Improved Short: {'total_queries': 116, 'top1_accuracy': np.float64(82.75862068965517), 'top3_recall': np.float64(88.79310344827587), 'top5_recall': np.float64(91.37931034482759)}

Baseline Long: {'total_queries': 41, 'top1_accuracy': np.float64(87.8048780487805), 'top3_recall': np.float64(95.1219512195122), 'top5_recall': np.float64(95.1219512195122)}
Improved Long: {'total_queries': 41, 'top1_accuracy': np.float64(90.2439024390244), 'top3_recall': np.float64(97.5609756097561), 'top5_recall': np.float64(100.0)}


In [ ]:
comparison = pd.DataFrame([
    {
        "Dataset": "Short (116)",
        "Baseline Top-1": baseline_short_metrics["top1_accuracy"],
        "Improved Top-1": improved_short_metrics["top1_accuracy"],
        "Δ Top-1": improved_short_metrics["top1_accuracy"] - baseline_short_metrics["top1_accuracy"],

        "Baseline Top-3": baseline_short_metrics["top3_recall"],
        "Improved Top-3": improved_short_metrics["top3_recall"],
        "Δ Top-3": improved_short_metrics["top3_recall"] - baseline_short_metrics["top3_recall"],

        "Baseline Top-5": baseline_short_metrics["top5_recall"],
        "Improved Top-5": improved_short_metrics["top5_recall"],
        "Δ Top-5": improved_short_metrics["top5_recall"] - baseline_short_metrics["top5_recall"],
    },
    {
        "Dataset": "Long (41)",
        "Baseline Top-1": baseline_long_metrics["top1_accuracy"],
        "Improved Top-1": improved_long_metrics["top1_accuracy"],
        "Δ Top-1": improved_long_metrics["top1_accuracy"] - baseline_long_metrics["top1_accuracy"],

        "Baseline Top-3": baseline_long_metrics["top3_recall"],
        "Improved Top-3": improved_long_metrics["top3_recall"],
        "Δ Top-3": improved_long_metrics["top3_recall"] - baseline_long_metrics["top3_recall"],

        "Baseline Top-5": baseline_long_metrics["top5_recall"],
        "Improved Top-5": improved_long_metrics["top5_recall"],
        "Δ Top-5": improved_long_metrics["top5_recall"] - baseline_long_metrics["top5_recall"],
    }
])

display(comparison.round(2))

,Dataset,Baseline Top-1,Improved Top-1,Δ Top-1,Baseline Top-3,Improved Top-3,Δ Top-3,Baseline Top-5,Improved Top-5,Δ Top-5
0,Short (117),78.45,82.76,4.31,87.93,88.79,0.86,89.66,91.38,1.72
1,Long (41),87.80,90.24,2.44,95.12,97.56,2.44,95.12,100.00,4.88


In [11]:
def compare_results(baseline_df, improved_df):

    merged = baseline_df[
        ["query_id", "query", "expected_document", "rank"]
    ].merge(
        improved_df[
            ["query_id", "query", "expected_document", "rank"]
        ],
        on=["query_id", "query", "expected_document"],
        suffixes=("_baseline", "_improved")
    )

    merged["baseline_rank_value"] = (
        merged["rank_baseline"].fillna(999)
    )

    merged["improved_rank_value"] = (
        merged["rank_improved"].fillna(999)
    )

    merged["rank_change"] = (
        merged["baseline_rank_value"]
        - merged["improved_rank_value"]
    )

    merged["status"] = np.select(
        [
            merged["rank_change"] > 0,
            merged["rank_change"] < 0
        ],
        [
            "Improved",
            "Regressed"
        ],
        default="Unchanged"
    )

    return merged

In [12]:
short_comparison = compare_results(
    baseline_short_details,
    improved_short_details
)

long_comparison = compare_results(
    baseline_long_details,
    improved_long_details
)

print("SHORT")
print(short_comparison["status"].value_counts())

print("\nLONG")
print(long_comparison["status"].value_counts())

SHORT
status
Unchanged    99
Improved     12
Regressed     5
Name: count, dtype: int64

LONG
status
Unchanged    36
Improved      3
Regressed     2
Name: count, dtype: int64


In [13]:
print("SHORT QUERY REGRESSIONS")

display(
    short_comparison[
        short_comparison["status"] == "Regressed"
    ][
        [
            "query",
            "expected_document",
            "rank_baseline",
            "rank_improved"
        ]
    ]
)

SHORT QUERY REGRESSIONS


,query,expected_document,rank_baseline,rank_improved
7,Do fathers get parental leave too or just moth...,New Parent Leave.md,1.0,2.0
9,"I need some time away, what are my options?",Vacation and Sick Leave.md,5.0,NaN
76,What does 'be better today than yesterday' mean?,Clef Values.md,1.0,3.0
81,What should I expect on my first day at Clef?,Welcome to Clef.md,1.0,2.0
83,How does Clef celebrate a new employee's first...,Welcome to Clef.md,2.0,5.0


In [14]:
print("SHORT QUERY IMPROVEMENTS")

display(
    short_comparison[
        short_comparison["status"] == "Improved"
    ][
        [
            "query",
            "expected_document",
            "rank_baseline",
            "rank_improved"
        ]
    ]
)

SHORT QUERY IMPROVEMENTS


,query,expected_document,rank_baseline,rank_improved
14,"I've been here 5 years, what's the deal with t...",Sabbatical.md,2.0,1.0
17,Does Clef pay for online courses and books?,Continuing Education.md,2.0,1.0
45,"I have a family emergency, what should I do?",Other Protected Absences.md,2.0,1.0
69,Can we have beer at a company celebration?,Drug and Alcohol Policy.md,2.0,1.0
70,Can Clef fire me without a reason?,At-Will Employment.md,NaN,1.0
86,Who is the CEO of Clef?,Handbook Introduction.md,3.0,2.0
95,What happens on Fridays in terms of team updates?,Communication and Transparency.md,2.0,1.0
98,Who do employees report to at Clef?,Direct Reports.md,2.0,1.0
99,What is Clef's core product value?,Product Manifesto.md,NaN,5.0
110,What tools does Clef use for policy discussions?,Policy Changes.md,4.0,3.0


In [15]:
print("LONG QUERY REGRESSIONS")

display(
    long_comparison[
        long_comparison["status"] == "Regressed"
    ][
        [
            "query",
            "expected_document",
            "rank_baseline",
            "rank_improved"
        ]
    ]
)

print("\nLONG QUERY IMPROVEMENTS")

display(
    long_comparison[
        long_comparison["status"] == "Improved"
    ][
        [
            "query",
            "expected_document",
            "rank_baseline",
            "rank_improved"
        ]
    ]
)

LONG QUERY REGRESSIONS


,query,expected_document,rank_baseline,rank_improved
20,When I'm working from home I know I should be ...,Working Remotely.md,2.0,5
21,I want to work from a coffee shop while travel...,Working Remotely.md,1.0,2



LONG QUERY IMPROVEMENTS


,query,expected_document,rank_baseline,rank_improved
3,I have a chronic illness that sometimes makes ...,Vacation and Sick Leave.md,NaN,3
10,Can I use a portion of my work hours during th...,Continuing Education.md,NaN,1
36,I need to schedule a meeting with a remote col...,Effective Meetings.md,2.0,1


In [16]:
comparison_DIR = BASE_DIR / "Comparison_Results"
comparison_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
comparison.to_json(
     comparison_DIR / "phase_4_retrieval_comparison.json",
    orient="records",
    indent=2
)

short_comparison.to_json(
    comparison_DIR / "short_queries_baseline_vs_improved.json",
    orient="records",
    indent=2,
    force_ascii=False
)

long_comparison.to_json(
    comparison_DIR / "long_queries_baseline_vs_improved.json",
    orient="records",
    indent=2,
    force_ascii=False
)

print("All comparison files saved.")

All comparison files saved.


In [20]:
print("=== PHASE 4 VALIDATION ===")

assert len(baseline_short_results) == 116
assert len(baseline_long_results) == 41

assert len(improved_short) == 116
assert len(improved_long) == 41

assert len(short_comparison) == 116
assert len(long_comparison) == 41

print("✓ Short baseline queries: 116")
print("✓ Long baseline queries: 41")
print("✓ Short improved queries: 116")
print("✓ Long improved queries: 41")
print("✓ Short comparison rows: 116")
print("✓ Long comparison rows: 41")

print("\nPhase 4 validation PASSED.")

=== PHASE 4 VALIDATION ===
✓ Short baseline queries: 116
✓ Long baseline queries: 41
✓ Short improved queries: 116
✓ Long improved queries: 41
✓ Short comparison rows: 116
✓ Long comparison rows: 41

Phase 4 validation PASSED.
